# PySpark 8: real-world use cases and interviews

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-08-real-world-and-interviews.ipynb)

The earlier modules taught the pieces. Production work and interviews ask you to combine them: choose a
design for a requirement, say what can fail, and prove the job is healthy.

This module walks through five common pipeline designs, runs the parts that can run locally (feature
scaling, a late-delivery backfill and a job entry point), shows how a job is packaged and submitted, and
ends with an interview question bank. For one executable pipeline that combines everything, open the
[end-to-end capstone](pyspark-concepts-end-to-end.ipynb).

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** PySpark modules [1](pyspark-01-foundations.ipynb) to [7](pyspark-07-monitoring-and-performance.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Nightly batch ELT into a partitioned lake

The nightly delivery is a full snapshot for one business date. Read the date from the job parameters,
validate that every row belongs to that date, write a candidate to a run-specific staging location, and
replace only that date in the target after the checks pass. A failed check leaves the published data as it
was. If the delivery contains only changes, merge by order key and source version instead of replacing the
date.

```python
business_date = args.business_date                      # "2026-10-04", from the scheduler
orders = spark.read.schema(order_schema).json(f"s3://acme-raw/orders/ds={business_date}/")
validate_delivery(orders, expected_date=business_date)   # raises if the contract is broken
orders.write.mode("errorifexists").parquet(f"s3://acme-staging/orders/run_id={run_id}/")
# reconcile staged counts and totals with the delivery manifest, then replace the date in the target
```

**Where it matters**
- **Daily snapshots** from an ERP or a vendor.
- **Reprocessing:** the same job with a different date parameter is the backfill.

**Watch out for**
- Deriving the business date from the wall clock. A rerun on another day writes the wrong partition.
- Publishing before reconciling. Record the delivery id, target date and committed table version for every run.

### Example

The backfill section below runs this pattern end to end on a small table: validate, replace one date with
dynamic partition overwrite, and rerun.

## Streaming from Kafka into a lakehouse table

A streaming job decodes Kafka events with a schema, applies the same transformations as the batch job, and
merges each micro-batch into a Delta or Iceberg table with `foreachBatch`. The checkpoint records offsets;
the merge uses a stable key and source version, because a micro-batch can be delivered twice after a
failure. Malformed events go to a quarantine with their topic, partition and offset so they can be
replayed. [PySpark 6](pyspark-06-structured-streaming.ipynb) shows the code and runs the file-based version.

**Where it matters**
- **Order and payment events** that dashboards need within minutes.
- **CDC topics** from Debezium applied to silver tables.

**Watch out for**
- An unbounded first batch on a backlog: set `maxOffsetsPerTrigger`.
- State without a watermark, and checkpoints shared between queries.

### Example

Health evidence for this design: input rows per second close to processing rows per second in
`lastProgress`, batch duration below the trigger interval, state size flat over a day, and quarantine counts
per hour within an agreed threshold.

## Feature engineering for machine learning

Feature tables join facts to small dimensions with broadcasts and compute features with built-in
expressions, so the same definitions run on years of history for training and on today's data for scoring.
A z-score needs the mean and standard deviation of the **whole** column: compute them once with an
aggregate and broadcast the single row back.

**Where it matters**
- **Training and scoring from one definition,** which reduces train/serve skew.
- **Customer-level aggregates** such as order counts and spend over a window.

**Watch out for**
- A Series-to-Series pandas UDF such as `lambda s: (s - s.mean()) / s.std()` runs once per Arrow batch, so it uses the batch mean, not the column mean.
- Recomputing the mean and standard deviation at scoring time. Save the training values and reuse them.

### Example

Compute spend per customer, then scale it with the column-wide mean and standard deviation broadcast from one aggregate row.

In [ ]:
spend = orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"), F.sum("amount").cast("double").alias("spend")
)
stats = spend.agg(F.mean("spend").alias("mu"), F.stddev("spend").alias("sigma"))
features = (
    spend.crossJoin(F.broadcast(stats))
    .withColumn("spend_z", (F.col("spend") - F.col("mu")) / F.col("sigma"))
    .drop("mu", "sigma")
)
features.orderBy("customer_id").show()

print("sum of z-scores:", round(sum(r.spend_z for r in features.collect()), 9))

**What to notice:** Customer 4, with the 1,180.00 bulk order, has by far the highest z-score. The z-scores sum to zero because
every row was scaled with the same column-wide mean and standard deviation: one aggregate row, broadcast
to every task.

## Log and event processing with messy input

Semi-structured logs break schema assumptions. Read with a declared schema in `PERMISSIVE` mode (the
corrupt-record example in [PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb)), write good rows to the
curated table and bad rows to a quarantine with a reason, land raw payloads as `VARIANT` or strings in bronze,
and promote fields to typed columns deliberately.

**Where it matters**
- **Application and web server logs** with optional fields that change by release.
- **Third-party webhooks** whose payloads you do not control.

**Watch out for**
- Failing the whole day for one bad line, or dropping bad lines silently.
- Exploding nested arrays with `explode` and losing events whose array is empty.

### Example

Why Spark here: quarantine plus schema-on-read lets ingestion continue while the producer is fixed. The
evidence is a reconciliation per file: landed lines = parsed rows + quarantined rows.

## Backfill a late delivery without duplicating data

A **backfill** rebuilds a historical slice from retained input. Name the affected business dates first, then
decide whether each delivery is a full replacement (overwrite the slice) or a set of changes (merge them).
With dynamic partition overwrite, writing only the affected dates replaces exactly those partitions.

**Where it matters**
- **A vendor sends Monday's missing orders on Wednesday:** rebuild Monday's silver data and gold totals and leave Tuesday alone.
- **A bug fix** that requires reprocessing a month.

**Watch out for**
- Dates whose correct result is now empty produce no rows, so dynamic overwrite never replaces them. Delete them explicitly.
- Other jobs writing the same slice concurrently.
- Using the delivery date instead of the business date, which puts late orders in the wrong day.

### Example

Monday has three orders and Tuesday two. A late delivery adds one Monday order. Rebuild Monday from all of its
input, check that Tuesday's files are untouched, and run the backfill twice.

In [ ]:
silver_path = os.path.join(WORK_DIR, "silver_orders_by_day")
ORDER_SCHEMA = "order_id int, business_date date, amount_cents long"
monday, tuesday = date(2026, 1, 12), date(2026, 1, 13)
delivered = spark.createDataFrame(
    [(1, monday, 1000), (2, monday, 2500), (3, monday, 500), (4, tuesday, 1200), (5, tuesday, 800)], ORDER_SCHEMA
)
late = spark.createDataFrame([(6, monday, 4000)], ORDER_SCHEMA)
delivered.write.mode("overwrite").partitionBy("business_date").parquet(silver_path)


def tuesday_files():
    folder = os.path.join(silver_path, "business_date=2026-01-13")
    return sorted(name for name in os.listdir(folder) if name.endswith(".parquet"))


def backfill(dates):
    """Rebuild the given business dates from every delivery that belongs to them."""
    rebuilt = delivered.unionByName(late).where(F.col("business_date").isin(dates))
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        rebuilt.write.mode("overwrite").partitionBy("business_date").parquet(silver_path)
    finally:
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")


def totals():
    day_totals = spark.read.parquet(silver_path).groupBy("business_date").agg(
        F.count("*").alias("orders"), F.sum("amount_cents").alias("cents")
    )
    return {str(r.business_date): (r.orders, r.cents) for r in day_totals.collect()}

print("before the backfill:", totals())
before_files = tuesday_files()
backfill([monday])
print("after the backfill :", totals())
backfill([monday])  # the scheduler retries the backfill
print("after a retry      :", totals())
print("Tuesday's files unchanged:", tuesday_files() == before_files)

**What to notice:** Monday goes from 3 orders (4,000 cents) to 4 orders (8,000 cents) after the late order is included, and the
retry leaves the same totals. Tuesday keeps the same files: dynamic overwrite replaced only the
`business_date=2026-01-12` partition.

## Package and submit a PySpark job

Notebooks are for exploration; production runs a Python entry point with parameters. `spark-submit` starts
the driver and requests executors from the cluster manager (YARN, Kubernetes or standalone). In **cluster**
deploy mode the driver runs inside the cluster, which is what schedulers use; in **client** mode it runs on
the machine that called `spark-submit`, which suits interactive work.

```bash
spark-submit \
  --master yarn --deploy-mode cluster \
  --num-executors 20 --executor-cores 4 --executor-memory 16g \
  --conf spark.sql.shuffle.partitions=400 \
  --archives pyspark_env.tar.gz#environment \
  --py-files orders_pipeline.zip \
  jobs/daily_orders.py --business-date 2026-10-04
```

Ship dependencies as a packed virtual environment (`--archives` with `venv-pack` or `conda-pack`) or a wheel
and `--py-files`; on Databricks, a job task with a wheel or a Python file does the same. Keep business logic
in importable functions so the entry point only parses arguments, builds the session and calls them.

**Where it matters**
- **Scheduled jobs** run by Airflow, Databricks Jobs or a Kubernetes operator.
- **CI:** the same functions are unit-tested with a local session.

**Watch out for**
- Hard-coded dates, paths or credentials. Pass parameters and read secrets from the platform's secret store.
- Driver-side `collect()` in cluster mode: the driver has its own, often small, memory limit.
- Python packages missing on executors cause `ModuleNotFoundError` only when a UDF first runs.

### Example

A job entry point: `main` parses arguments and calls a pure function, so a scheduler, a test and this notebook
run the same code. Here it is called directly instead of through `spark-submit`.

In [ ]:
import argparse


def daily_revenue(orders_df, business_date):
    """Business logic: importable and testable, with no I/O."""
    return (
        orders_df.where(F.to_date("order_ts") == F.lit(business_date).cast("date"))
        .where(F.col("status").isin("paid", "shipped"))
        .agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))
        .withColumn("business_date", F.lit(business_date).cast("date"))
    )


def main(argv):
    parser = argparse.ArgumentParser(description="Publish daily revenue for one business date.")
    parser.add_argument("--business-date", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args(argv)
    result = daily_revenue(orders, args.business_date)  # a real job reads its input table here
    result.write.mode("overwrite").parquet(os.path.join(args.output, f"business_date={args.business_date}"))
    return result.first()

published = main(["--business-date", "2026-01-12", "--output", os.path.join(WORK_DIR, "gold_daily_revenue")])
print(published)

**What to notice:** On 12 January, order 9 (shipped, 1,180.00) and order 10 (paid, 74.00) count as revenue: 2 orders, 1,254.00.
The same `daily_revenue` function can be unit-tested with a small DataFrame, and `main` is what
`spark-submit` would call with the scheduler's arguments.

## Interview question bank

A strong answer names the **API**, the **execution model** and the **production concern**. For "How do you
deduplicate orders?": "`row_number()` over `Window.partitionBy("order_id").orderBy(source_lsn desc)`, keep
`rn = 1`, then `MERGE`. That forces one shuffle by key; if one key is hot the window skews, so I check
partition sizes and let AQE split it or salt it." A 30-second skeleton:

```text
1. API     - the exact functions and window spec you call
2. Engine  - narrow vs wide, where the shuffle is, partition count
3. Ops     - idempotency, skew, cost, monitoring, failure recovery
```

Say each answer out loud before reading it. Interviewers probe the plan, so tie every method to a shuffle, a
partition count or a failure mode.

### Fundamentals

**Q. What is lazy evaluation, and why does Spark use it?**

Transformations only build a plan; an action triggers execution. Waiting lets Catalyst optimize the whole query (push filters down, prune columns, pick join strategies) instead of running each line. See [PySpark 1](pyspark-01-foundations.ipynb).

**Q. What is the difference between narrow and wide transformations?**

Narrow transformations (`filter`, `select`, `withColumn`) work inside each partition. Wide ones (`groupBy`, joins, `distinct`, windows with `partitionBy`) need rows with the same key together, so they shuffle and start a new stage.

**Q. How do you handle malformed records when reading files?**

Choose the read mode on purpose: usually `PERMISSIVE` with `columnNameOfCorruptRecord`, so bad rows are kept, quarantined with a reason and counted; `FAILFAST` when any bad row must stop the job; `DROPMALFORMED` almost never. Reconcile input rows with valid plus rejected rows. See [PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb).

**Q. How do you keep the latest record per key?**

`row_number()` over `Window.partitionBy(key).orderBy(updated_at desc, tie_breaker desc)` and keep `rn = 1`. Without a tie-breaker the winner among equal timestamps can change between runs; `dropDuplicates` gives no control over which row survives. See [PySpark 4](pyspark-04-windows-and-nested-data.ipynb).

**Q. Why are Python UDFs slow, and what do you use instead?**

Rows are serialized between the JVM and a Python worker and hidden from the optimizer. Use built-in functions first, then higher-order array functions, then a vectorized pandas UDF. See [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb).

### Performance

**Q. What is a shuffle, and why is it expensive?**

Rows are redistributed so equal keys land together: written to local disk, sent over the network, read back and often sorted, with a stage boundary. Every `Exchange` in a plan is one. See [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**Q. When does a broadcast join help, and what are its limits?**

When one side is small enough to copy to every executor: the large side is joined in place with no shuffle. The limit is driver and executor memory; it does not fix skew on the large side. Confirm `BroadcastHashJoin` in the plan. See [PySpark 3](pyspark-03-joins-and-aggregations.ipynb).

**Q. How do you handle data skew, and what can AQE not fix?**

Spot it as one task far slower than its peers with much more shuffle read. Let AQE split skewed join partitions first; then broadcast the small side, salt the hot key, or fix the source (often a null or placeholder key). AQE works at shuffle boundaries, so it cannot fix a skewed scan.

**Q. How do you choose partition columns and avoid small files?**

Partition on a low-cardinality column almost every query filters on (usually a date), aiming for hundreds of MB per partition. Control file counts at write time with `repartition(partition_col)` and compact with the table format.

**Q. When should you cache, and how is that different from checkpointing?**

Cache a DataFrame reused by several actions and `unpersist` it after. Caching keeps the lineage; `checkpoint()` writes to reliable storage and cuts the lineage, which suits very long plans.

**Q. How do you debug a slow Spark job?**

Find the longest stage, then its slowest task, and compare input and spill with its peers. Read `explain()` for unexpected exchanges, sort-merge joins that should broadcast, and scans with no partition filters. Change one thing and measure again.

### Pipelines and operations

**Q. Walk me through a bronze, silver, gold pipeline.**

Bronze is raw data with an explicit schema and lineage columns, with bad rows quarantined. Silver is cleaned, deduplicated and conformed to one row per business key. Gold is shaped for consumers. Each layer is rebuildable from the one before, so a bug fix is a rerun. See the [capstone](pyspark-concepts-end-to-end.ipynb).

**Q. How do you make a batch job idempotent?**

Overwrite exactly the slice the batch owns (dynamic partition overwrite or a `MERGE` on the key) instead of appending, make dedupe deterministic, and keep wall-clock values out of the data. Test by running the batch twice and comparing.

**Q. How do you build a slowly changing dimension type 2?**

Compare incoming attributes with the current row using null-safe equality; close changed current rows at the change date, insert new current rows, and assert exactly one current row per key. On Delta this is one `MERGE`. See [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb).

**Q. How is exactly-once achieved in Structured Streaming?**

A replayable source with offsets, a durable checkpoint of offsets and state, and an idempotent or transactional sink (Delta, or a `foreachBatch` `MERGE`). A watermark bounds state for late data. See [PySpark 6](pyspark-06-structured-streaming.ipynb).

**Q. Client or cluster deploy mode for a scheduled job?**

Cluster mode: the driver runs inside the cluster, so the job does not depend on the scheduler machine staying up, and driver resources are managed with the executors.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A full-snapshot delivery and a change delivery arrive for the same table. How do the writes differ?

**A.** The snapshot replaces its business date (overwrite the slice). The change delivery is merged by key and source version, because it does not contain the unchanged rows.

**Q2.** Why should a job take the business date as a parameter?

**A.** So a rerun or a backfill on another day writes the same slice. Deriving it from the clock makes output depend on when the job ran.

**Q3.** Why broadcast the mean and standard deviation instead of using a pandas UDF for z-scores?

**A.** A Series-to-Series pandas UDF sees one Arrow batch at a time, so its mean is the batch mean. A single aggregate row broadcast back uses the whole column.

**Q4.** What evidence shows a backfill is correct?

**A.** Per-date counts and totals that match the retained input, untouched files in partitions outside the backfill, and the same result after a second run.

**Q5.** Why keep business logic out of the job's entry point?

**A.** So the same functions are unit-tested locally and called by the scheduler; the entry point only parses arguments, builds the session and calls them.

## Key takeaways

- Name the API, the execution model and the production concern in every answer.
- Full snapshots replace a business date; change deliveries merge by key and version.
- Backfills reuse the daily job with a date parameter and are proven by per-date reconciliation and a rerun.
- Compute global statistics with an aggregate, never inside a pandas UDF.
- Production jobs are parameterized entry points around tested functions, submitted in cluster mode.

**Next:** [PySpark concepts, end to end](pyspark-concepts-end-to-end.ipynb), then the [PySpark coding challenges](pyspark-coding-challenges.ipynb).